# BioCLIP-2 Preprocessing Fix Test (GitHub open_clip)

Direct test using open_clip cloned from GitHub.
Tests the fix for preprocess_train vs preprocess_val issue.

## Cell 1: Check GPU

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}GB")

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"\nUsing device: {device}")

## Cell 2: Clone open_clip from GitHub

In [ ]:
import os
import sys
import subprocess

open_clip_path = "/tmp/open_clip"

if not os.path.exists(open_clip_path):
    print(f"Cloning open_clip from GitHub...")
    result = subprocess.run(
        ["git", "clone", "https://github.com/mlfoundations/open_clip.git", open_clip_path],
        capture_output=True,
        text=True
    )
    if result.returncode == 0:
        print(f"✅ Cloned to {open_clip_path}")
        # Install in development mode so imports work
        print(f"Installing in development mode...")
        os.system(f'pip install -q -e "{open_clip_path}"')
        print(f"✅ Installed with pip install -e")
    else:
        print(f"Clone output: {result.stderr}")
        print(f"Will use pip installed version")
else:
    print(f"✅ open_clip already exists at {open_clip_path}")

# Make sure it's installed via pip as well (most reliable)
print(f"\nEnsuring open_clip is installed via pip...")
os.system('pip install -q open-clip-torch')
print(f"✅ open_clip installed")

# Add to path as well
sys.path.insert(0, open_clip_path)
print(f"✅ Python path configured")

## Cell 3: Import and verify

In [ ]:
import open_clip

print(f"✅ open_clip imported successfully")
print(f"open_clip version: {open_clip.__version__ if hasattr(open_clip, '__version__') else 'unknown'}")
print(f"open_clip location: {open_clip.__file__}")

# Check which version we're using
if '/tmp/open_clip' in open_clip.__file__:
    print(f"✅ Using GitHub version from /tmp/open_clip")
else:
    print(f"✅ Using pip installed version (which is fine!)")

## Cell 4: Load BioCLIP-2

In [ ]:
print("\n" + "="*70)
print("Loading BioCLIP-2 from imageomics")
print("="*70)

model_id = "imageomics/bioclip-2"
hub_model_id = f"hf-hub:{model_id}"

print(f"\nModel ID: {hub_model_id}")
print("Creating model and transforms...")

model, preprocess_train, preprocess_val = open_clip.create_model_and_transforms(hub_model_id)
tokenizer = open_clip.get_tokenizer(hub_model_id)

model = model.to(device)
model.eval()

print(f"✅ BioCLIP-2 loaded on {device}")
print(f"\nModel type: {type(model).__name__}")
print(f"Preprocess train type: {type(preprocess_train).__name__}")
print(f"Preprocess val type: {type(preprocess_val).__name__}")
print(f"Tokenizer type: {type(tokenizer).__name__}")

## Cell 5: Text encoding test

In [ ]:
print("\n" + "="*70)
print("Text Encoding Test")
print("="*70)

prompts = ["a photo of grape", "a photo of potato", "a photo of tomato", "a photo of strawberry"]
print(f"\nPrompts:")
for i, p in enumerate(prompts, 1):
    print(f"  {i}. {p}")

tokens = tokenizer(prompts).to(device)
print(f"\nTokens shape: {tokens.shape}")

with torch.no_grad():
    text_embeds = model.encode_text(tokens)
    print(f"\nText embeddings shape: {text_embeds.shape}")
    print(f"Before normalization:")
    print(f"  Norms: {text_embeds.norm(dim=-1)}")
    
    text_embeds = text_embeds / text_embeds.norm(dim=-1, keepdim=True)
    print(f"After normalization:")
    print(f"  Norms: {text_embeds.norm(dim=-1)}")

print(f"\n✅ Text encoding successful")

## Cell 6: Download test image

In [ ]:
from PIL import Image
import urllib.request

print("\nDownloading strawberry image...")

strawberry_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/2/29/Fragaria_x_ananassa_Foto_by_CF_Weise.jpg/800px-Fragaria_x_ananassa_Foto_by_CF_Weise.jpg"
image_path = "/tmp/strawberry.jpg"

try:
    urllib.request.urlretrieve(strawberry_url, image_path, timeout=10)
    image = Image.open(image_path).convert('RGB')
    print(f"✅ Image downloaded: {image.size}")
except Exception as e:
    print(f"Download failed ({e}), using red test image")
    image = Image.new('RGB', (224, 224), color=(200, 0, 0))
    print(f"✅ Using red test image: {image.size}")

## Cell 7: Critical Test - preprocess_train vs preprocess_val

In [ ]:
print("\n" + "="*70)
print("CRITICAL TEST: preprocess_train vs preprocess_val")
print("="*70)
print("\nTesting both preprocessing methods on the SAME image.")
print("If they give different results, we found the bug!\n")

results = {}

for preprocess_name, preprocess_fn in [("TRAIN (with augmentation)", preprocess_train), 
                                         ("VAL (no augmentation)", preprocess_val)]:
    print("\n" + "-"*70)
    print(f"Using: preprocess_{preprocess_name}")
    print("-"*70)
    
    # Preprocess image
    image_tensor = preprocess_fn(image).unsqueeze(0).to(device)
    
    print(f"\nImage tensor after preprocessing:")
    print(f"  Shape: {image_tensor.shape}")
    print(f"  DType: {image_tensor.dtype}")
    print(f"  Min/Max: [{image_tensor.min():.4f}, {image_tensor.max():.4f}]")
    print(f"  Mean/Std: {image_tensor.mean():.4f} / {image_tensor.std():.4f}")
    
    # Encode image
    with torch.no_grad():
        image_embeds = model.encode_image(image_tensor)
        
        print(f"\nImage embeddings (before normalization):")
        norm_before = image_embeds.norm(dim=-1).item()
        print(f"  Shape: {image_embeds.shape}")
        print(f"  Norm: {norm_before:.4f}")
        
        # Normalize embeddings
        image_embeds = image_embeds / image_embeds.norm(dim=-1, keepdim=True)
        
        print(f"\nImage embeddings (after normalization):")
        print(f"  Norm: {image_embeds.norm(dim=-1).item():.4f}")
        
        # Compute similarity scores
        logits = image_embeds @ text_embeds.T
        probs = torch.softmax(logits, dim=-1)
        
        print(f"\nSimilarity logits: {logits.squeeze()}")
        print(f"Probabilities:     {probs.squeeze()}")
        
        # Get top prediction
        pred_idx = torch.argmax(probs)
        pred_conf = probs[0, pred_idx].item()
        pred_label = prompts[pred_idx].replace("a photo of ", "")
        
        print(f"\n🎯 Prediction: {pred_label} ({pred_conf:.1%})")
        
        results[preprocess_name] = {
            'label': pred_label,
            'confidence': pred_conf,
            'probs': probs.squeeze().cpu(),
            'norm_before': norm_before,
            'logits': logits.squeeze().cpu()
        }

## Cell 8: Analysis

In [ ]:
print("\n" + "="*70)
print("ANALYSIS - Did the fix work?")
print("="*70)

train_result = results["TRAIN (with augmentation)"]
val_result = results["VAL (no augmentation)"]

train_label = train_result['label']
train_conf = train_result['confidence']
train_probs = train_result['probs']
train_norm = train_result['norm_before']

val_label = val_result['label']
val_conf = val_result['confidence']
val_probs = val_result['probs']
val_norm = val_result['norm_before']

print(f"\npreprocess_TRAIN (with augmentation):")
print(f"  Prediction: {train_label} ({train_conf:.1%})")
print(f"  Probabilities: {train_probs}")
print(f"  Embedding norm: {train_norm:.4f}")

print(f"\npreprocess_VAL (no augmentation):")
print(f"  Prediction: {val_label} ({val_conf:.1%})")
print(f"  Probabilities: {val_probs}")
print(f"  Embedding norm: {val_norm:.4f}")

# Check if random (all ~25%)
train_is_random = (train_probs - 0.25).abs().max() < 0.02
val_is_random = (val_probs - 0.25).abs().max() < 0.02

print(f"\nRandom check (all probs ~0.25):")
print(f"  preprocess_train is random: {train_is_random}")
print(f"  preprocess_val is random: {val_is_random}")

print("\n" + "="*70)
print("RESULT")
print("="*70)

if train_is_random and val_conf > 0.4:
    print(f"\n✅✅✅ FIX CONFIRMED! ✅✅✅")
    print(f"\npreprocess_train (old code) shows RANDOM 25% per class")
    print(f"  This was the BUG in the old code!")
    print(f"\npreprocess_val (new code) shows CORRECT {val_conf:.1%} for {val_label}")
    print(f"  This is FIXED!")
    print(f"\nConclusion: Using preprocess_val in the code fixes the issue.")
    
elif val_is_random and not train_is_random:
    print(f"\n⚠️ INVERTED - val is random, train is correct")
    print(f"  Maybe we need to use preprocess_train after all?")
    
elif train_is_random and val_is_random:
    print(f"\n❌ STILL BROKEN - Both show random 25% per class")
    print(f"  Problem is not preprocessing, it's something else.")
    
elif val_conf > train_conf:
    print(f"\n✅ preprocess_val is BETTER than preprocess_train")
    print(f"  Confidence improvement: {val_conf - train_conf:+.1%}")
    
else:
    print(f"\n⚠️ UNCLEAR - Need to check output manually")

## Cell 9: Summary

In [ ]:
print("\n" + "="*70)
print("SUMMARY")
print("="*70)

print(f"\nTest Environment:")
print(f"  Device: {device}")
print(f"  open_clip source: GitHub (from /tmp/open_clip)")
print(f"  BioCLIP model: {model_id}")
print(f"  Test image: Strawberry (or red substitute)")

print(f"\nResults:")
print(f"  preprocess_train: {train_label} ({train_conf:.1%})")
print(f"  preprocess_val: {val_label} ({val_conf:.1%})")

print(f"\nWhat to check:")
print(f"1. Is preprocess_val showing >50% confidence?")
print(f"2. Is preprocess_train showing ~25% (random)?")
print(f"3. Are the embedding norms > 1.0 before normalization?")
print(f"4. Are the logits significantly different (not all ~0.63)?")

print(f"\nNext step:")
print(f"If preprocess_val is working well, the fix in vlm_pipeline.py is correct.")
print(f"Make sure the code uses preprocess_val, not preprocess_train!")